# 68 Optode co-registration invariance

For each subject, run `ColoredStickerProcessor` independently on the original and on the anonymized mesh, match the detected sticker centres between the two runs by nearest neighbour, and report per-optode Euclidean deviations for both the raw sticker centres and the scalp-projected optode positions (sticker centre minus optode length along the surface normal).

Populates Table 4.5 of the thesis. Expected outcome: zero up to numerical noise, because every sticker sits on the cap which lies outside the facial mask by construction.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    SUBJECTS, subject_paths, load_raw, load_anon, load_landmarks,
    available_subjects, missing_report, run_pipeline,
)

import numpy as np
import pandas as pd

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

import cedalion

from cedalion.geometry.photogrammetry.processors import (

    ColoredStickerProcessor,

)

from scipy.spatial import KDTree



# Yellow optode stickers, same HSV range as notebook 41.

COLORS = {'O': (0.11, 0.21, 0.7, 1.0)}

OPTODE_LENGTH = 22.6 * cedalion.units.mm

## 1. Sticker detection wrapper

Returns sticker centres and normals as plain numpy arrays for easy nearest-neighbour matching.

In [ ]:
def detect_stickers(surface):

    proc = ColoredStickerProcessor(colors=COLORS)

    centres, normals = proc.process(surface, details=False)

    c_np = centres.pint.dequantify().values

    n_np = normals.values if hasattr(normals, 'values') else np.asarray(normals)

    return c_np, n_np



def match_by_nn(a, b):

    '''For each row of a, return the index of the nearest row of b.'''

    tree = KDTree(b)

    _, idx = tree.query(a)

    return idx

## 2. Per-subject comparison

Only the optode cohort (S1--S7) carries cap-mounted stickers; bare-cap subjects have nothing to detect, so they are skipped.

In [ ]:
from _thesis_helpers import OPTODE_SUBJECTS

rows = []

for n in OPTODE_SUBJECTS:

    paths = subject_paths(n)

    if not (paths.raw_exists and paths.anon_exists):

        print(f'skipping Subject{n}: missing scans')

        continue

    print(f'--- Subject{n} ---')

    surface_orig = load_raw(n)

    surface_anon = load_anon(n)



    c_orig, n_orig = detect_stickers(surface_orig)

    c_anon, n_anon = detect_stickers(surface_anon)



    if len(c_orig) == 0 or len(c_anon) == 0:

        print(f'  no stickers detected on one side '

              f'(orig={len(c_orig)}, anon={len(c_anon)})')

        continue



    idx = match_by_nn(c_orig, c_anon)

    sticker_dev = np.linalg.norm(c_orig - c_anon[idx], axis=1)



    L = float(OPTODE_LENGTH.to('mm').magnitude)

    scalp_orig = c_orig - L * n_orig

    scalp_anon = c_anon[idx] - L * n_anon[idx]

    scalp_dev = np.linalg.norm(scalp_orig - scalp_anon, axis=1)



    rows.append({

        'subject': n,

        'n_stickers_original': len(c_orig),

        'n_stickers_anonymized': len(c_anon),

        'n_matched': len(idx),

        'sticker_mean_mm': float(sticker_dev.mean()),

        'sticker_median_mm': float(np.median(sticker_dev)),

        'sticker_max_mm': float(sticker_dev.max()),

        'scalp_mean_mm': float(scalp_dev.mean()),

        'scalp_median_mm': float(np.median(scalp_dev)),

        'scalp_max_mm': float(scalp_dev.max()),

    })

    print(rows[-1])

## 3. Summary

In [ ]:
df = pd.DataFrame(rows).sort_values('subject').reset_index(drop=True)

df

## 4. Save

In [ ]:
out = OUT_DIR / 'coreg_invariance.csv'

df.to_csv(out, index=False)

print(f'Wrote {out}')